# Sensitivity check: extended regularisation grid for the frozen-feature classifiers (about 15–30 min, no GPU)
In three of the ten Phikon partitions the selected C lay at the edge of the grid {0.001, 0.01, 0.1, 1}. This notebook re-fits the
logistic-regression classifiers of **Phikon, ImageNet ViT-B/16 and ImageNet EfficientNetB0** with an extended grid
{0.0001, 0.001, 0.01, 0.1, 1, 10, 100}, using the **features already saved on Google Drive** (nothing is re-extracted), and reports
whether the selected C and the internal and external accuracy change.

**How to run (Google Colab):** a CPU runtime is enough. *Runtime → Run all*, allow Google Drive access. It needs the folders
`MyDrive/extra_results/foundation_model` (Phikon features) and `MyDrive/imagenet_frozen_results` from the earlier runs.
At the end a SUMMARY is printed and `extended_c_grid_results.zip` is downloaded. **Send both to Claude.**

In [ ]:
# %% Settings
import os
TEST_MODE = os.environ.get("GS_TEST_MODE") == "1"     # offline test only; leave as is
SEEDS      = [42, 43, 44, 45, 46]
TILE       = 400
N_BOOT     = 2000
FM_BATCH   = 64
BACKBONES  = {"ViT-B16_ImageNet": "google/vit-base-patch16-224",      # ImageNet-21k pretrained, ImageNet-1k fine-tuned, 224 px
              "EfficientNetB0_ImageNet": "keras:EfficientNetB0"}       # Keras ImageNet weights (backbone of the paper's CNNs)
GROUPS_COMMIT = "912f342fe2a781e847f395d79982dad4d3d425ae"
GROUPS_SHA256 = "2ff9e992f759f265dfc9aa2363de0dfcc8e6559884d8bb02a4c925214fdf7e23"
GROUPS_URL = f"https://raw.githubusercontent.com/GeorgeBatch/LC25000-clean/{GROUPS_COMMIT}/kaggle/lc25000_image_groups.csv"
LH_URL     = "https://ndownloader.figshare.com/files/45206104"
IM_DIR     = "/content/drive/MyDrive/imagenet_frozen_results"
SAVE_DIR   = "/content/drive/MyDrive/extended_c_grid_results"
REF_DIR    = "/content/drive/MyDrive/extra_results"            # earlier Phikon run (splits + external probabilities), optional
DATA_DIR   = None
LH_ROOT    = "/content/lunghist700"
if TEST_MODE:
    N_BOOT, SEEDS, FM_BATCH = 100, [42, 43], 8
    DATA_DIR, LH_ROOT, SAVE_DIR = os.environ["GS_DATA"], os.environ["GS_LH"], os.environ["GS_OUT"]
    IM_DIR = os.environ["GS_IM"]
    REF_DIR = os.environ.get("GS_REF", "/nonexistent")
    GROUPS_URL = os.environ["GS_GROUPS"]

In [ ]:
# %% Imports and shared helpers
import json, time, hashlib, random, glob, re, subprocess, zipfile, platform
import numpy as np, pandas as pd, cv2
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
CLASSES = ["lung_aca", "lung_n", "lung_scc"]
if not TEST_MODE:
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(SAVE_DIR, exist_ok=True)

def ece_score(p, y, n_bins=15):
    conf, pred = p.max(1), p.argmax(1); bins, e = np.linspace(0, 1, n_bins + 1), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def cluster_ci(correct, clusters, seed=0):
    rng = np.random.default_rng(seed); u = np.unique(clusters); idx = {k: np.where(clusters == k)[0] for k in u}; a = []
    for _ in range(N_BOOT):
        s = np.concatenate([idx[k] for k in rng.choice(u, len(u), replace=True)]); a.append(correct[s].mean())
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def metrics(p, y, clusters):
    yp = p.argmax(1); cm = confusion_matrix(y, yp, labels=[0, 1, 2]); rec = cm.diagonal() / cm.sum(1).clip(min=1)
    return {"n": int(len(y)), "accuracy": float((yp == y).mean()),
            "accuracy_95ci_cluster": cluster_ci((yp == y).astype(float), clusters),
            "macro_f1": float(f1_score(y, yp, average="macro")), "balanced_accuracy": float(balanced_accuracy_score(y, yp)),
            "ece": ece_score(p, y), "brier": float(((p - np.eye(3)[y]) ** 2).sum(1).mean()),
            "recall": dict(zip(CLASSES, rec.round(4).tolist())),
            "auc_ovr_macro": float(roc_auc_score(y, p, multi_class="ovr")) if len(np.unique(y)) == 3 else None,
            "errors": int((yp != y).sum()), "confusion_matrix": cm.tolist()}

def agg(vals):
    v = np.array(vals, float); m = v.mean(); sd = v.std(ddof=1) if len(v) > 1 else 0.0
    return {"mean": float(m), "sd": float(sd), "min": float(v.min()), "max": float(v.max()), "per_seed": v.round(5).tolist()}

In [ ]:
# %% LC25000 (MD5-deduplicated) + source-tile groups + both partitioning protocols (identical to the earlier notebooks)
def find_lung_dir(root):
    for d, ds, _ in os.walk(root):
        if all(c in ds for c in CLASSES): return d
    raise FileNotFoundError(root)
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = find_lung_dir(kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images"))
else:
    DATA_DIR = find_lung_dir(DATA_DIR)
rows = []
for label, c in enumerate(CLASSES):
    for f in sorted(glob.glob(os.path.join(DATA_DIR, c, "*"))):
        if f.lower().endswith((".jpg", ".jpeg", ".png")):
            with open(f, "rb") as fh:
                rows.append({"path": f, "label": label, "cls": c, "filename": os.path.basename(f), "md5": hashlib.md5(fh.read()).hexdigest()})
df_all = pd.DataFrame(rows)
df = df_all.drop_duplicates("md5", keep="first").reset_index(drop=True)
conflict = df_all.groupby("md5")["label"].nunique()
df = df[~df["md5"].isin(conflict[conflict > 1].index)].reset_index(drop=True)
groups = pd.read_csv(GROUPS_URL)
if not TEST_MODE:
    import urllib.request
    assert hashlib.sha256(urllib.request.urlopen(GROUPS_URL).read()).hexdigest() == GROUPS_SHA256, "grouping file changed - send this message to Claude"
groups = groups[groups["label"].isin(CLASSES)][["filename", "label", "group_id"]].rename(columns={"label": "g_label"})
df = df.merge(groups, on="filename", how="left")
assert df["group_id"].notna().all(), "images without group - send this message to Claude"
assert (df["g_label"] == df["cls"]).all()
df["group_id"] = df["group_id"].astype(int)
print(f"LC25000 lung images kept: {len(df)} | groups: {df['group_id'].nunique()}")

def image_split(df, seed):
    """Protocol A: conventional stratified image-level 70/15/15 split (seed 42 reproduces the first benchmark)."""
    tr, tmp = train_test_split(df, test_size=0.30, stratify=df["label"], random_state=seed)
    va, te = train_test_split(tmp, test_size=0.50, stratify=tmp["label"], random_state=seed)
    parts = {"train": tr.reset_index(drop=True), "val": va.reset_index(drop=True), "test": te.reset_index(drop=True)}
    for a in parts:
        for b in parts:
            if a < b: assert not set(parts[a]["md5"]) & set(parts[b]["md5"]), "hash overlap!"
    return parts

def group_split(df, seed, fr=(0.70, 0.15, 0.15)):
    """Protocol B: identical to the group-split notebook (whole source groups, largest relative shortfall)."""
    rng = np.random.default_rng(seed); assign = {}; keys = ["train", "val", "test"]
    for c in CLASSES:
        sizes = df[df["cls"] == c].groupby("group_id").size()
        g = sizes.index.values.copy(); rng.shuffle(g)
        target = np.array(fr) * sizes.sum(); filled = np.zeros(3)
        for gid in g:
            k = int(np.argmax((target - filled) / target)); assign[gid] = keys[k]; filled[k] += sizes[gid]
    s = df.assign(split=df["group_id"].map(assign))
    parts = {k: s[s["split"] == k].reset_index(drop=True) for k in keys}
    for a in parts:
        for b in parts:
            if a < b:
                assert not set(parts[a]["group_id"]) & set(parts[b]["group_id"]), "group overlap!"
                assert not set(parts[a]["md5"]) & set(parts[b]["md5"]), "hash overlap!"
    return parts

SPLITS = {("A", s): image_split(df, s) for s in SEEDS}
SPLITS.update({("B", s): group_split(df, s) for s in SEEDS})
leak = {s: float(100 * SPLITS[("A", s)]["test"]["group_id"].isin(set(SPLITS[("A", s)]["train"]["group_id"])).mean()) for s in SEEDS}
print("Protocol A leakage (% of test images whose source group is in training):", leak)
for (p, s), parts in SPLITS.items():
    pass

# Check that the partitions are identical to those of the paper (if the earlier run is on Drive)
n_checked = 0
for (p, s), parts in SPLITS.items():
    d = os.path.join(REF_DIR, f"splits/protocol{p}_seed{s}")
    for k in ["train", "val", "test"]:
        f = os.path.join(d, f"{k}.csv")
        if os.path.exists(f):
            ref = pd.read_csv(f)
            assert sorted(ref["filename"]) == sorted(parts[k]["filename"]), f"partition differs: protocol {p} seed {s} {k} - send this message to Claude"
            n_checked += 1
print(f"Partitions checked against the earlier run: {n_checked} of {3 * len(SPLITS)} files identical"
      + ("" if n_checked else " (earlier splits not found on Drive - partitions are regenerated with the same code and seeds)"))

In [ ]:
# %% LungHist700 index (external only; no tuning)
if not glob.glob(LH_ROOT + "/**/*.jp*g", recursive=True):
    os.makedirs(LH_ROOT, exist_ok=True); rar = "/content/LungHist700.rar"
    if not os.path.exists(rar): subprocess.run(["wget", "-q", "-O", rar, LH_URL], check=True)
    if subprocess.run(["which", "unrar"], capture_output=True).returncode != 0:
        subprocess.run("apt-get -qq install -y unrar > /dev/null", shell=True)
    if subprocess.run(["unrar", "x", "-o+", "-inul", rar, LH_ROOT + "/"]).returncode != 0:
        subprocess.run(["7z", "x", "-y", f"-o{LH_ROOT}", rar], check=True, stdout=subprocess.DEVNULL)
rows = []
for f in sorted(glob.glob(LH_ROOT + "/**/*.jp*g", recursive=True)):
    m = re.match(r"^(aca|nor|scc)(?:_(bd|md|pd))?_(20x|40x)_(\d+)\.jpe?g$", os.path.basename(f).lower())
    if m:
        sup, sub, res_, iid = m.groups()
        rows.append({"path": f, "superclass": sup, "subclass": sub or "", "resolution": res_, "image_id": int(iid),
                     "label": {"aca": 0, "nor": 1, "scc": 2}[sup]})
ext = pd.DataFrame(rows)
for c in glob.glob(LH_ROOT + "/**/*.csv", recursive=True):
    meta = pd.read_csv(c)
    if {"superclass", "resolution", "image_id", "patient_id"} <= set(meta.columns):
        meta = meta.copy()
        if "subclass" not in meta: meta["subclass"] = ""
        meta["subclass"] = meta["subclass"].fillna("").astype(str).str.lower().replace("nan", "")
        meta["superclass"] = meta["superclass"].astype(str).str.lower(); meta["resolution"] = meta["resolution"].astype(str).str.lower()
        meta["image_id"] = meta["image_id"].astype(int)
        ext = ext.merge(meta[["superclass", "subclass", "resolution", "image_id", "patient_id"]],
                        on=["superclass", "subclass", "resolution", "image_id"], how="left"); break
assert "patient_id" in ext and ext["patient_id"].notna().all(), "patient IDs missing - send this message to Claude"
y_ext, pat = ext["label"].values, ext["patient_id"].values
T = 12
print(f"LungHist700: {len(ext)} images, {ext['patient_id'].nunique()} patients")

def ext_tiles(path):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB); H, W = img.shape[:2]
    out = [img[yy:yy + TILE, xx:xx + TILE] for yy in range(0, H - TILE + 1, TILE) for xx in range(0, W - TILE + 1, TILE)]
    assert len(out) == T, f"{path}: {len(out)} tiles instead of 12 - send this message to Claude"
    return out

def ext_result(p):
    res = metrics(p, y_ext, pat)
    res["by_resolution"] = {r: float((p.argmax(1)[ext["resolution"] == r] == y_ext[ext["resolution"] == r]).mean()) for r in ["20x", "40x"]}
    pp = ext[["patient_id", "label"]].assign(**{f"p{i}": p[:, i] for i in range(3)}).groupby(["patient_id", "label"], as_index=False).mean()
    res["patient_class_accuracy"] = float((pp[["p0", "p1", "p2"]].to_numpy().argmax(1) == pp["label"].to_numpy()).mean())
    res["patient_class_units"] = int(len(pp))
    return res

In [ ]:
# %% Load the cached frozen features (Phikon from the earlier run, ImageNet controls from notebook 6)
FEAT = {}
srcs = {"Phikon": os.path.join(REF_DIR, "foundation_model"),
        "ViT-B16_ImageNet": os.path.join(IM_DIR, "ViT-B16_ImageNet"), "EfficientNetB0_ImageNet": os.path.join(IM_DIR, "EfficientNetB0_ImageNet")}
for name, d in srcs.items():
    fi, fe = os.path.join(d, "features_lc25000.npy"), os.path.join(d, "features_lunghist700_tiles.npy")
    assert os.path.exists(fi) and os.path.exists(fe), f"{name}: cached features not found in {d} - send this message to Claude"
    FEAT[name] = (np.load(fi), np.load(fe))
    assert len(FEAT[name][0]) == len(df) and len(FEAT[name][1]) == len(ext) * T, f"{name}: feature count mismatch - send this message to Claude"
    print(f"{name}: {FEAT[name][0].shape}, {FEAT[name][1].shape}")

In [ ]:
# %% Re-fit with the extended C grid (same standardisation, validation-only selection, single test evaluation)
row_of = {p: i for i, p in enumerate(df["path"])}
C_GRID = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]
os.makedirs(SAVE_DIR, exist_ok=True)
def fit_probe(X, Xt, parts):
    itr, iva, ite = [[row_of[p] for p in parts[k]["path"]] for k in ["train", "val", "test"]]
    sc = StandardScaler().fit(X[itr]); best = None; vals = {}
    for C in C_GRID:
        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(X[itr]), df["label"].values[itr])
        acc = float((clf.predict(sc.transform(X[iva])) == df["label"].values[iva]).mean()); vals[C] = acc
        if best is None or acc > best[0]: best = (acc, C, clf)
    _, C, clf = best
    return C, vals, clf.predict_proba(sc.transform(X[ite])), clf.predict_proba(sc.transform(Xt)).reshape(len(ext), T, 3).mean(1)
for name, (X, Xt) in FEAT.items():
    for (prot, s), parts in SPLITS.items():
        out = os.path.join(SAVE_DIR, f"{name}_protocol{prot}_seed{s}.json")
        if os.path.exists(out): continue
        C, vals, p_test, p_ext = fit_probe(X, Xt, parts)
        res = {"backbone": name, "protocol": prot, "seed": s, "C": C, "val_accuracy_by_C": {str(k): v for k, v in vals.items()},
               "C_at_grid_boundary": C in (C_GRID[0], C_GRID[-1]),
               "internal": metrics(p_test, parts["test"]["label"].values, parts["test"]["group_id"].values), "external": ext_result(p_ext)}
        json.dump(res, open(out, "w"), indent=1)
        print(f"{name} | Protocol {prot} seed {s}: C={C} internal {100*res['internal']['accuracy']:.2f}% | external {100*res['external']['accuracy']:.1f}%")

In [ ]:
# %% SUMMARY (copy everything printed below and send it to Claude together with extended_c_grid_results.zip)
print("=" * 78 + "\nSUMMARY - extended C grid", C_GRID)
for name in FEAT:
    for prot in ["A", "B"]:
        R = [json.load(open(os.path.join(SAVE_DIR, f"{name}_protocol{prot}_seed{s}.json"))) for s in SEEDS]
        ia = [100 * r["internal"]["accuracy"] for r in R]; ea = [100 * r["external"]["accuracy"] for r in R]
        print(f"{name:24s} Protocol {prot}: C {[r['C'] for r in R]} | boundary {[r['C_at_grid_boundary'] for r in R]} | "
              f"internal {np.mean(ia):.2f} ± {np.std(ia, ddof=1):.2f} per seed {np.round(ia, 2).tolist()} | "
              f"external {np.mean(ea):.1f} ± {np.std(ea, ddof=1):.1f} per seed {np.round(ea, 1).tolist()}")
print("=" * 78)
if not TEST_MODE:
    subprocess.run(f"cd {os.path.dirname(SAVE_DIR)} && zip -qr /content/extended_c_grid_results.zip {os.path.basename(SAVE_DIR)}", shell=True)
    from google.colab import files; files.download("/content/extended_c_grid_results.zip")